# DB Looker

按資料表分別讀取 SQLite 中所有資料列。


In [ ]:
from pathlib import Path
import sqlite3

DB_PATH = Path('./xchain_demo.db')
assert DB_PATH.exists(), f"找不到資料庫檔案: {DB_PATH.resolve()}"

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
print(f"Connected: {DB_PATH.resolve()}")

# 列出目前資料庫中的所有資料表名稱
def list_all_tables():
    cursor = conn.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name ASC")
    all_tables = [row[0] for row in cursor.fetchall()]
    return all_tables

def read_all_rows(table_name: str):
    cursor = conn.execute(f"SELECT * FROM {table_name}")
    rows = cursor.fetchall()
    columns = [desc[0] for desc in cursor.description or []]
    return columns, rows

def print_table(table_name: str):
    count_cursor = conn.execute(f"SELECT COUNT(*) FROM {table_name}")
    total_rows = count_cursor.fetchone()[0] or 0

    if total_rows <= 10:
        data_cursor = conn.execute(f"SELECT * FROM {table_name} ORDER BY rowid ASC")
        rows = data_cursor.fetchall()
        columns = [desc[0] for desc in data_cursor.description or []]
    else:
        data_cursor = conn.execute(
            f"SELECT * FROM {table_name} ORDER BY rowid DESC LIMIT 10"
        )
        rows = data_cursor.fetchall()
        rows.reverse()
        columns = [desc[0] for desc in data_cursor.description or []]

    print(f"\n=== {table_name} ===")
    print(f"columns: {columns}")
    print(f"row_count: {total_rows}")
    for row in rows:
        print(dict(row))

def clear_table(table_name: str):
    conn.execute(f"DELETE FROM {table_name}")
    conn.commit()
    print(f"Deleted all rows from {table_name}")


In [ ]:
# List All Tables

list_all_tables()

In [ ]:
## Read Tables

# print_table("xchain_txs")
print_table("xchain_timeline_events")
# print_table("raw_logs")
# print_table("indexer_cursors")
# print_table("search_index")
# print_table("risk_reports")

In [ ]:
## Clear Tables

clear_table("xchain_txs")
clear_table("xchain_timeline_events")
clear_table("raw_logs")
clear_table("indexer_cursors")
clear_table("search_index")
clear_table("risk_reports")


In [ ]:
# 使用完可關閉連線
conn.close()
print("DB connection closed")
